In [1]:
!pip install llama-index-llms-ollama

In [3]:
import os
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama
from typing import List, Dict

In [16]:
def run_few_shot(context:str , prompt:str , examples:List[Dict[str,str]]) -> str:
    llm=Ollama(model="gemma2:2b",temperature=0.7 )
    examples_str = "\n".join(
        f"Example {i+1}:\nContext:\n{ex.get('context','')}\n"
        f"Question: {ex.get('question','')}\n"
        f"Answer: {ex.get('answer','')}\n"
        for i, ex in enumerate(examples)
    )
    template_str = (
        "You are an expert AI assistant.\n"
        "Use ONLY the provided context to answer the user's question. "
        "If the context is insufficient or does not mention the answer, reply exactly: "
        "'Not enough information.'\n\n"
        "Follow the style and reasoning illustrated by the examples.\n\n"
        "Examples:\n{examples_str}\n"
        "--- End of Examples ---\n\n"
        "Context:\n{context_str}\n\n"
        "User Question: {query_str}\n\n"
        "Answering Rules:\n"
        "1) Be concise and precise (3–6 sentences, unless the question requires more).\n"
        "2) Use bullet points for lists.\n"
        "3) At the end, include a 'Sources:' section with short snippets or filenames from the context you used.\n\n"
        "Final Answer:"
    )
    prompt_template = PromptTemplate(template_str).format(
        examples_str=examples_str,context_str=context,query_str=prompt)
    response=llm.complete(
        prompt=prompt_template
    )
    return response.text
    


In [17]:
# Few-Shot: Add examples so the model mimics your style
shots = [
    {
        "context": "Positional encodings inject order information into sequences.",
        "question": "Why are positional encodings needed?",
        "answer": (
            "They give the model a sense of word order.\n"
            "- Without them, the model treats tokens as a bag of words.\n"
            "- Encodings ensure the sequence structure is preserved.\n"
            "Sources: lecture_notes.txt"
        )
    },
    {
        "context": "Multi-head attention projects queries, keys, and values into multiple subspaces.",
        "question": "What is the benefit of multi-head attention?",
        "answer": (
            "It lets the model learn from different representation subspaces.\n"
            "- Captures diverse relationships.\n"
            "- Improves contextual understanding.\n"
            "Sources: attention_paper.pdf"
        )
    },
]

context_text = (
    "Context from attention_mechanism.pdf"
    "In the attention mechanism, softmax is used on the similarity scores "
    "between queries and keys to produce attention weights."
)

query_text = "What does softmax do in attention?"

In [18]:
ansk = run_few_shot(context=context_text, prompt=query_text, examples=shots)

print(ansk)

In the attention mechanism, softmax is applied to the similarity scores between queries and keys.  This process generates attention weights that indicate the importance of different parts of the input sequence for generating the output. The scores are transformed into probabilities through softmax function.  The resulting attention weights represent the likelihood that each query will attend to a specific key in the sequence, allowing the model to focus on relevant parts of the input while generating the output. 

Sources:
* attention_mechanism.pdf 

